In [ ]:
!pip install batchgenerators

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 4.5 MB/s eta 0:00:00
  Created wheel for batchgenerators: filename=batchgenerators-0.25.1-py3-none-any.whl size=93088 sha256=89e721f114c1bc2121cdd3199889f4f522307bfdd1bd19f1cb0ed538d801de96
  Stored in directory: /root/.cache/pip/wheels/56/11/c7/fadca30e054c602093ffe36ba8a2f0a87dd2f86ac75191d3ed
Successfully built batchgenerators


In [ ]:
!git clone https://github.com/mohammadnabia/DA_nnUNet.git
%cd DA_nnUNet
!pip install -e .

Cloning into 'DA_nnUNet'...
remote: Enumerating objects: 297, done.
remote: Counting objects: 100% (297/297), done.
remote: Compressing objects: 100% (270/270), done.
remote: Total 297 (delta 41), reused 267 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (297/297), 2.58 MiB | 13.07 MiB/s, done.
Resolving deltas: 100% (41/41), done.
/content/DA_nnUNet
Obtaining file:///content/DA_nnUNet
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Using cached argparse-1.4.0-py2.py3-none-any.whl.metadata (2.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 29.8 MB/s eta 0:00:00
  

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!unzip -q "/content/drive/MyDrive/BraTS-peds2023/BraTS-PEDs-2023.zip" -d /content/BraTS_PEDs_2023

In [ ]:
import json
import os
import shutil

split_file = "/content/drive/MyDrive/100_epoch_training_on_bratspeds_finetune/splits_final.json"
with open(split_file, 'r') as f:
    splits = json.load(f)

val_cases = splits[0]['val']  # Fold 0

source_dir = "/content/BraTS_PEDs_2023/ASNR-MICCAI-BraTS2023-PED-Challenge-TrainingData"
labels_dir = "/content/labelsTs_eval"
os.makedirs(labels_dir, exist_ok=True)

for case_id in val_cases:
    case_path = os.path.join(source_dir, case_id)
    seg_file = os.path.join(case_path, f"{case_id}-seg.nii.gz")
    if os.path.exists(seg_file):
        shutil.copy(seg_file, os.path.join(labels_dir, f"{case_id}.nii.gz"))
    else:
        print(f"Segmentation missing for case: {case_id}")


In [ ]:
print("Number of segmentation files:", len(os.listdir(labels_dir)))


Number of segmentation files: 20


In [ ]:
!ls /content/drive/MyDrive/nnUNet_Pruned_Results_peds2023/BraTS-PEDS_FT100epoch_80pruned_noTTA

BraTS-PED-00008-000.nii.gz  BraTS-PED-00099-000.nii.gz
BraTS-PED-00021-000.nii.gz  BraTS-PED-00101-000.nii.gz
BraTS-PED-00026-000.nii.gz  BraTS-PED-00104-000.nii.gz
BraTS-PED-00042-000.nii.gz  BraTS-PED-00107-000.nii.gz
BraTS-PED-00050-000.nii.gz  BraTS-PED-00110-000.nii.gz
BraTS-PED-00055-000.nii.gz  BraTS-PED-00115-000.nii.gz
BraTS-PED-00063-000.nii.gz  BraTS-PED-00118-000.nii.gz
BraTS-PED-00078-000.nii.gz  BraTS-PED-00132-000.nii.gz
BraTS-PED-00079-000.nii.gz  dataset.json
BraTS-PED-00084-000.nii.gz  plans.json
BraTS-PED-00086-000.nii.gz  predict_from_raw_data_args.json
BraTS-PED-00096-000.nii.gz


In [ ]:
predictions_dir = "/content/drive/MyDrive/nnUNet_Pruned_Results_peds2023/BraTS-PEDS_FT100epoch_80pruned_noTTA"


In [ ]:
import SimpleITK as sitk

def compute_metrics(pred_path, gt_path):
    pred = sitk.ReadImage(pred_path)
    gt = sitk.ReadImage(gt_path)

    pred = sitk.Cast(pred, sitk.sitkUInt8)
    gt = sitk.Cast(gt, sitk.sitkUInt8)

    dice_filter = sitk.LabelOverlapMeasuresImageFilter()
    dice_filter.Execute(pred, gt)
    dice_score = dice_filter.GetDiceCoefficient()

    hausdorff_filter = sitk.HausdorffDistanceImageFilter()
    hausdorff_filter.Execute(pred, gt)
    hausdorff_distance = hausdorff_filter.GetHausdorffDistance()

    return dice_score, hausdorff_distance


In [ ]:
import os
import SimpleITK as sitk
import numpy as np
import pandas as pd

# مسیر ground truth segmentation
gt_dir = "/content/labelsTs_eval"

# مسیر پیش‌بینی‌های pruned
pred_dir = "/content/drive/MyDrive/nnUNet_Pruned_Results_peds2023/BraTS-PEDS_FT100epoch_80pruned_noTTA"

dice_list = []
hausdorff_list = []
case_names = []

for filename in os.listdir(gt_dir):
    if filename.endswith(".nii.gz"):
        gt_path = os.path.join(gt_dir, filename)
        pred_path = os.path.join(pred_dir, filename)
        if not os.path.exists(pred_path):
            print(f"Prediction missing for case: {filename}")
            continue

        dice, haus = compute_metrics(pred_path, gt_path)
        dice_list.append(dice)
        hausdorff_list.append(haus)
        case_names.append(filename)

        print(f"{filename} - Dice: {dice:.4f} - Hausdorff: {haus:.4f}")

# ذخیره نتایج در یک CSV
results = pd.DataFrame({
    "Case": case_names,
    "Dice": dice_list,
    "Hausdorff": hausdorff_list
})
results.to_csv("evaluation_results.csv", index=False)
print("Results saved to evaluation_results.csv")


BraTS-PED-00026-000.nii.gz - Dice: 0.3836 - Hausdorff: 50.2593
BraTS-PED-00096-000.nii.gz - Dice: 0.7031 - Hausdorff: 36.9053
BraTS-PED-00132-000.nii.gz - Dice: 0.8155 - Hausdorff: 59.8415
BraTS-PED-00078-000.nii.gz - Dice: 0.8497 - Hausdorff: 64.4127
BraTS-PED-00101-000.nii.gz - Dice: 0.7929 - Hausdorff: 49.3356
BraTS-PED-00050-000.nii.gz - Dice: 0.6171 - Hausdorff: 40.5586
BraTS-PED-00063-000.nii.gz - Dice: 0.7327 - Hausdorff: 81.9390
BraTS-PED-00115-000.nii.gz - Dice: 0.1650 - Hausdorff: 49.0102
BraTS-PED-00008-000.nii.gz - Dice: 0.7168 - Hausdorff: 104.5084
BraTS-PED-00107-000.nii.gz - Dice: 0.9487 - Hausdorff: 3.0000
BraTS-PED-00118-000.nii.gz - Dice: 0.8062 - Hausdorff: 28.7924
BraTS-PED-00042-000.nii.gz - Dice: 0.5810 - Hausdorff: 52.1920
BraTS-PED-00099-000.nii.gz - Dice: 0.7820 - Hausdorff: 60.7536
BraTS-PED-00086-000.nii.gz - Dice: 0.7928 - Hausdorff: 52.7352
BraTS-PED-00110-000.nii.gz - Dice: 0.4456 - Hausdorff: 57.4891
BraTS-PED-00079-000.nii.gz - Dice: 0.9103 - Hausdorff: 

In [ ]:
import SimpleITK as sitk
import numpy as np

def compute_metrics_per_class(pred_path, gt_path, label_mapping):
    pred = sitk.ReadImage(pred_path)
    gt = sitk.ReadImage(gt_path)

    pred = sitk.GetArrayFromImage(pred)
    gt = sitk.GetArrayFromImage(gt)

    results = {}
    for class_name, labels in label_mapping.items():
        pred_mask = np.isin(pred, labels).astype(np.uint8)
        gt_mask = np.isin(gt, labels).astype(np.uint8)

        if np.sum(gt_mask) == 0 and np.sum(pred_mask) == 0:
            # هیچکدوم وجود ندارن
            dice_score = 1.0
            hausdorff_distance = 0.0
        elif np.sum(gt_mask) == 0 or np.sum(pred_mask) == 0:
            # یکی از دو مورد وجود نداره
            dice_score = 0.0
            hausdorff_distance = np.nan
        else:
            pred_mask = sitk.GetImageFromArray(pred_mask)
            pred_mask.CopyInformation(sitk.ReadImage(pred_path))
            gt_mask = sitk.GetImageFromArray(gt_mask)
            gt_mask.CopyInformation(sitk.ReadImage(gt_path))

            dice_filter = sitk.LabelOverlapMeasuresImageFilter()
            dice_filter.Execute(pred_mask, gt_mask)
            dice_score = dice_filter.GetDiceCoefficient()

            hausdorff_filter = sitk.HausdorffDistanceImageFilter()
            hausdorff_filter.Execute(pred_mask, gt_mask)
            hausdorff_distance = hausdorff_filter.GetHausdorffDistance()

        results[class_name] = (dice_score, hausdorff_distance)

    return results


In [ ]:
print(gt_path)
print(pred_path)


/content/labelsTs_eval/BraTS-PED-00021-000.nii.gz
/content/drive/MyDrive/nnUNet_Pruned_Results_peds2023/BraTS-PEDS_FT100epoch_80pruned_noTTA/BraTS-PED-00021-000.nii.gz


In [ ]:
import os
import pandas as pd

# دیکشنری برچسب‌ها (از dataset.json)
label_mapping = {
    "whole tumor": [1, 2, 3],
    "tumor core": [2, 3],
    "enhancing tumor": [3]
}

# آدرس‌ها
pred_folder = "/content/drive/MyDrive/nnUNet_Pruned_Results_peds2023/BraTS-PEDS_FT100epoch_80pruned_noTTA"
gt_folder = "/content/labelsTs_eval"

# ساختار دیتافریم خروجی
results_list = []

# لیست کیس‌ها (بر اساس Ground Truth)
gt_cases = [f for f in os.listdir(gt_folder) if f.endswith('.nii.gz')]

for gt_file in gt_cases:
    case_id = gt_file.replace('.nii.gz', '')
    pred_file = case_id + ".nii.gz"   # تغییر این خط
    pred_path = os.path.join(pred_folder, pred_file)
    gt_path = os.path.join(gt_folder, gt_file)

    if not os.path.exists(pred_path):
        print(f"❌ پیش‌بینی برای {case_id} پیدا نشد.")
        continue

    metrics = compute_metrics_per_class(pred_path, gt_path, label_mapping)
    for class_name, (dice, hausdorff) in metrics.items():
        results_list.append({
            "case_id": case_id,
            "class": class_name,
            "dice": dice,
            "hausdorff": hausdorff
        })


# ساخت دیتافریم و ذخیره نتایج
df = pd.DataFrame(results_list)
output_path = "/content/evaluation_results_per_class.csv"
df.to_csv(output_path, index=False)
print(f"✅ نتایج ذخیره شد: {output_path}")


✅ نتایج ذخیره شد: /content/evaluation_results_per_class.csv


In [ ]:
summary = df.groupby('class').agg({
    'dice': ['mean', 'median'],
    'hausdorff': ['mean', 'median']
}).reset_index()

# بازنویسی نام ستون‌ها برای نمایش
summary.columns = ['Class', 'Mean Dice', 'Median Dice', 'Mean HD95', 'Median HD95']
print(summary)


             Class  Mean Dice  Median Dice  Mean HD95  Median HD95
0  enhancing tumor   0.381548     0.314147  50.781648    53.094256
1       tumor core   0.330331     0.236593  50.274689    44.163345
2      whole tumor   0.810675     0.888627  52.959183    52.463570


In [ ]:
from tabulate import tabulate

print(tabulate(summary, headers='keys', tablefmt='pretty', showindex=False))


+-----------------+---------------------+---------------------+-------------------+--------------------+
|      Class      |      Mean Dice      |     Median Dice     |     Mean HD95     |    Median HD95     |
+-----------------+---------------------+---------------------+-------------------+--------------------+
| enhancing tumor | 0.38154765633172566 | 0.31414669760514613 | 50.78164816869045 | 53.094255809833136 |
|   tumor core    | 0.3303310708321209  | 0.23659332555843976 | 50.27468901120768 | 44.16334479227005  |
|   whole tumor   | 0.8106748224775243  | 0.8886274699762873  | 52.9591827468657  |  52.4635704482702  |
+-----------------+---------------------+---------------------+-------------------+--------------------+


In [ ]:
mean_dice_overall = np.nanmean(df['dice'])
mean_hausdorff_overall = np.nanmean(df['hausdorff'])

print(f"✅ میانگین کلی Dice: {mean_dice_overall:.4f}")
print(f"✅ میانگین کلی Hausdorff: {mean_hausdorff_overall:.4f}")


✅ میانگین کلی Dice: 0.5075
✅ میانگین کلی Hausdorff: 51.4076
